### Import & Data Load


In [1]:
# Import & Data Load

In [2]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder

In [3]:
import os

# 실행 경로와 상관없이 데이터를 로드할 수 있도록 상대 경로 자동 조정
data_dir = "." if os.path.exists("train.csv") else "../.."
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

In [4]:
train.head(5)

,ID,gender,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,activity,smoke_status,medical_history,family_medical_history,sleep_pattern,edu_level,mean_working,stress_score
0,TRAIN_0000,F,72,161.49,58.47,279.84,165,100,143.35,0.87,moderate,ex-smoker,high blood pressure,diabetes,sleep difficulty,bachelors degree,NaN,0.63
1,TRAIN_0001,M,88,179.87,77.60,257.37,178,111,146.94,0.07,moderate,ex-smoker,NaN,diabetes,normal,graduate degree,NaN,0.83
2,TRAIN_0002,M,47,182.47,89.93,226.66,134,95,142.61,1.18,light,ex-smoker,NaN,NaN,normal,high school diploma,9.0,0.70
3,TRAIN_0003,M,69,185.78,68.63,206.74,158,92,137.26,0.48,intense,ex-smoker,high blood pressure,NaN,oversleeping,graduate degree,NaN,0.17
4,TRAIN_0004,F,81,164.63,71.53,255.92,171,116,129.37,0.34,moderate,ex-smoker,diabetes,diabetes,sleep difficulty,bachelors degree,NaN,0.36


### Check Data


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        3000 non-null   object 
 1   gender                    3000 non-null   object 
 2   age                       3000 non-null   int64  
 3   height                    3000 non-null   float64
 4   weight                    3000 non-null   float64
 5   cholesterol               3000 non-null   float64
 6   systolic_blood_pressure   3000 non-null   int64  
 7   diastolic_blood_pressure  3000 non-null   int64  
 8   glucose                   3000 non-null   float64
 9   bone_density              3000 non-null   float64
 10  activity                  3000 non-null   object 
 11  smoke_status              3000 non-null   object 
 12  medical_history           1711 non-null   object 
 13  family_medical_history    1514 non-null   object 
 14  sleep_pa

In [6]:
train.isnull().sum()

ID                             0
gender                         0
age                            0
height                         0
weight                         0
cholesterol                    0
systolic_blood_pressure        0
diastolic_blood_pressure       0
glucose                        0
bone_density                   0
activity                       0
smoke_status                   0
medical_history             1289
family_medical_history      1486
sleep_pattern                  0
edu_level                    607
mean_working                1032
stress_score                   0
dtype: int64

In [7]:
# 결측값 있는 칼럼(column) 확인
missing_columns_train = train.columns[train.isnull().sum() > 0]
missing_columns_train

Index(['medical_history', 'family_medical_history', 'edu_level',
       'mean_working'],
      dtype='object')

In [8]:
train[missing_columns_train].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   medical_history         1711 non-null   object 
 1   family_medical_history  1514 non-null   object 
 2   edu_level               2393 non-null   object 
 3   mean_working            1968 non-null   float64
dtypes: float64(1), object(3)
memory usage: 93.9+ KB


In [9]:
# ==========================================
# 1. 통합 결측치 처리 (Data Cleaning)
# ==========================================
for df in [train, test]:
    df["medical_history"] = df["medical_history"].fillna("none")
    df["family_medical_history"] = df["family_medical_history"].fillna("none")
    df["edu_level"] = df["edu_level"].fillna("Unknown")
    df["is_working_missing"] = df["mean_working"].isnull().astype(int)

# 평균 근무 시간 연령대별 중앙값 대치 (Train 기준)
train["age_group"] = (train["age"] // 10) * 10
test["age_group"] = (test["age"] // 10) * 10
age_working_medians = train.groupby("age_group")["mean_working"].median()
overall_median = train["mean_working"].median()

for df in [train, test]:
    df["mean_working"] = df.apply(
        lambda row: (
            age_working_medians.get(row["age_group"], overall_median)
            if pd.isnull(row["mean_working"])
            else row["mean_working"]
        ),
        axis=1,
    )
    df.drop("age_group", axis=1, inplace=True)


# ==========================================
# 2. 파생 피처 및 복합 질환 해체 (Feature Engineering)
# ==========================================
for df in [train, test]:
    df["bmi"] = df["weight"] / ((df["height"] / 100) ** 2)
    df["pulse_pressure"] = (
        df["systolic_blood_pressure"] - df["diastolic_blood_pressure"]
    )
    df["map"] = (df["systolic_blood_pressure"] + 2 * df["diastolic_blood_pressure"]) / 3
    df["is_extreme_overwork"] = (df["mean_working"] >= 12).astype(int)
    df["is_low_bone_density"] = (df["bone_density"] <= -1.0).astype(int)
    df["is_high_pulse_pressure"] = (df["pulse_pressure"] > 80).astype(int)
    df["overwork_and_poor_sleep"] = (
        (df["is_extreme_overwork"] == 1) & (df["sleep_pattern"] == "sleep difficulty")
    ).astype(int)
    df["vascular_bone_risk"] = (
        (df["is_low_bone_density"] == 1) & (df["is_high_pulse_pressure"] == 1)
    ).astype(int)
    df["glucose_cholesterol_ratio"] = df["glucose"] / (df["cholesterol"] + 1)

# 복합 질환(medical_history) 동적 이진화 플래그 생성
diseases = set()
for col in ["medical_history", "family_medical_history"]:
    for val in pd.concat([train[col], test[col]]).unique():
        for d in val.split(","):
            diseases.add(d.strip())
diseases.discard("none")
diseases = sorted(list(diseases))

for col, prefix in [("medical_history", "med"), ("family_medical_history", "fam")]:
    for disease in diseases:
        feat_name = f'{prefix}_{disease.lower().replace(" ", "_")}'
        train[feat_name] = train[col].apply(
            lambda x: 1 if disease in [d.strip() for d in x.split(",")] else 0
        )
        test[feat_name] = test[col].apply(
            lambda x: 1 if disease in [d.strip() for d in x.split(",")] else 0
        )


# ==========================================
# 3. 범주형 데이터 변환 및 불필요 컬럼 제거 (Encoding & Drop)
# ==========================================
edu_map = {
    "Unknown": 0,
    "high school diploma": 1,
    "bachelors degree": 2,
    "graduate degree": 3,
}
activity_map = {"light": 1, "moderate": 2, "intense": 3}

for df in [train, test]:
    df["edu_level_encoded"] = df["edu_level"].map(edu_map)
    df["activity_encoded"] = df["activity"].map(activity_map)

# 사용이 끝난 원본 문자열 컬럼 일괄 삭제
drop_cols = ["edu_level", "activity", "medical_history", "family_medical_history"]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

In [10]:
# 순서형 인코딩된 컬럼을 제외한 순수 범주형 데이터 변환 및 타입 지정
categorical_cols = ["gender", "smoke_status", "sleep_pattern"]

for col in categorical_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")

In [11]:
x_train = train.drop(["ID", "stress_score"], axis=1)
y_train = train["stress_score"]

x_test = test.drop("ID", axis=1)

In [12]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import RobustScaler
import warnings

warnings.filterwarnings("ignore")


def objective(trial):
    params = {
        "objective": "regression_l1",
        "metric": "mae",
        "random_state": 42,
        "verbose": -1,
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    mae_scores = []

    for train_idx, val_idx in kf.split(x_train, y_train):
        X_tr, y_tr = x_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
        X_val, y_val = x_train.iloc[val_idx].copy(), y_train.iloc[val_idx]

        # 루프 내부에서 매번 Scaler 객체를 신규 생성하여 폴드간 정보 전이(Data Leakage) 차단
        scaler_g = RobustScaler()
        scaler_c = RobustScaler()

        g_tr = scaler_g.fit_transform(X_tr[["glucose"]])
        c_tr = scaler_c.fit_transform(X_tr[["cholesterol"]])
        X_tr["metabolic_load_index"] = (g_tr + c_tr).flatten()

        g_val = scaler_g.transform(X_val[["glucose"]])
        c_val = scaler_c.transform(X_val[["cholesterol"]])
        X_val["metabolic_load_index"] = (g_val + c_val).flatten()

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
        )

        preds = model.predict(X_val)
        mae = mean_absolute_error(y_val, preds)
        mae_scores.append(mae)

    return np.mean(mae_scores)


# Coarse-to-fine 튜닝을 위한 1차 20회 탐색 진행 (MAE 기준)
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print(f"Best Trial MAE: {study.best_value:.6f}")
print(f"Best Params: {study.best_params}")

# 최적 파라미터로 최종 5-Fold 모델 학습 및 테스트 예측
best_params = study.best_params
best_params["objective"] = "regression_l1"
best_params["metric"] = "mae"
best_params["random_state"] = 42
best_params["verbose"] = -1

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(x_train))
test_preds = np.zeros(len(x_test))
mae_list = []

for fold, (train_idx, val_idx) in enumerate(kf.split(x_train, y_train)):
    X_tr, y_tr = x_train.iloc[train_idx].copy(), y_train.iloc[train_idx]
    X_val, y_val = x_train.iloc[val_idx].copy(), y_train.iloc[val_idx]
    x_te_fold = x_test.copy()

    scaler_g = RobustScaler()
    scaler_c = RobustScaler()

    g_tr = scaler_g.fit_transform(X_tr[["glucose"]])
    c_tr = scaler_c.fit_transform(X_tr[["cholesterol"]])
    X_tr["metabolic_load_index"] = (g_tr + c_tr).flatten()

    g_val = scaler_g.transform(X_val[["glucose"]])
    c_val = scaler_c.transform(X_val[["cholesterol"]])
    X_val["metabolic_load_index"] = (g_val + c_val).flatten()

    g_te = scaler_g.transform(x_te_fold[["glucose"]])
    c_te = scaler_c.transform(x_te_fold[["cholesterol"]])
    x_te_fold["metabolic_load_index"] = (g_te + c_te).flatten()

    model = lgb.LGBMRegressor(**best_params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )

    val_preds = model.predict(X_val)
    oof_preds[val_idx] = val_preds
    mae = mean_absolute_error(y_val, val_preds)
    mae_list.append(mae)

    test_preds += model.predict(x_te_fold) / 5
    print(f"Fold {fold+1} MAE: {mae:.6f}")

print(f"==> 5-Fold Average MAE: {np.mean(mae_list):.6f}")

[I 2026-06-16 20:33:42,080] A new study created in memory with name: no-name-81ef69ca-eaaa-4d25-9880-dd4d58ee072d
[I 2026-06-16 20:34:51,368] Trial 0 finished with value: 0.1898610793568129 and parameters: {'n_estimators': 627, 'learning_rate': 0.08839574228639116, 'num_leaves': 107, 'min_child_samples': 17, 'subsample': 0.6848203290478208, 'colsample_bytree': 0.9365196290696928, 'reg_alpha': 0.10384162811702208, 'reg_lambda': 0.03422721780451811}. Best is trial 0 with value: 0.1898610793568129.
[I 2026-06-16 20:35:28,001] Trial 1 finished with value: 0.20038402832434477 and parameters: {'n_estimators': 488, 'learning_rate': 0.024300626494671605, 'num_leaves': 62, 'min_child_samples': 13, 'subsample': 0.6472680017786886, 'colsample_bytree': 0.6306353857785707, 'reg_alpha': 0.012293563847922866, 'reg_lambda': 0.7430562873778294}. Best is trial 0 with value: 0.1898610793568129.
[I 2026-06-16 20:36:10,396] Trial 2 finished with value: 0.20226887811711655 and parameters: {'n_estimators': 6

Best Trial MAE: 0.184199
Best Params: {'n_estimators': 686, 'learning_rate': 0.06928895029017235, 'num_leaves': 94, 'min_child_samples': 11, 'subsample': 0.7914084158654671, 'colsample_bytree': 0.9573255832919458, 'reg_alpha': 0.11045268062326417, 'reg_lambda': 0.22193761732868833}
Fold 1 MAE: 0.174980
Fold 2 MAE: 0.181548
Fold 3 MAE: 0.189159
Fold 4 MAE: 0.195037
Fold 5 MAE: 0.180273
==> 5-Fold Average MAE: 0.184199


In [13]:
submission = pd.read_csv(os.path.join(data_dir, "sample_submission.csv"))

In [14]:
submission["stress_score"] = test_preds
submission.head()

,ID,stress_score
0,TEST_0000,0.439998
1,TEST_0001,0.738794
2,TEST_0002,0.274363
3,TEST_0003,0.410800
4,TEST_0004,0.573286


In [15]:
# 결과 저장 경로 자동 대응
output_dir = "result/v3" if os.path.exists("result/v3") else "."
submission.to_csv(os.path.join(output_dir, "submit_03.csv"), index=False)